In [2]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║  LightGBM + ELO  │  Predict 2023 NCAA Tournament                   ║
║                                                                      ║
║  Identical data pipeline + ELO implementation to XGBoost+ELO 2023  ║
║  model — only the model is swapped to LightGBM for direct           ║
║  apples-to-apples comparison.                                        ║
║                                                                      ║
║  TRAINING DATA (labels = game outcomes):                             ║
║    ① Regular season games  2013–2023  → each game = one training row ║
║       features = game-level box-score diffs + running ELO at        ║
║       the time of the game (updated game-by-game chronologically)   ║
║    ② Tournament games      2013–2022  → each game = one training row ║
║       features = season-agg reg-season stats + seeds +              ║
║       end-of-regular-season ELO snapshot for each team              ║
║                                                                      ║
║  TEST DATA (never touched during training):                          ║
║    2023 Tournament matchups only                                     ║
║    Features: 2023 reg-season agg stats + 2023 seeds +               ║
║              2023 end-of-regular-season ELO snapshots               ║
║                                                                      ║
║  ELO IMPLEMENTATION                                                  ║
║    • Each team resets to 1500 at the start of every season          ║
║    • Updated chronologically through regular-season games only       ║
║    • K-factor: K=30 if |MOV| > 10 pts (blowout), K=20 otherwise    ║
║    • Reg-season rows: ELO snapshot BEFORE that game plays           ║
║    • Tournament rows: ELO snapshot at END of regular season         ║
║    • 4 ELO features: ELO_A, ELO_B, ELO_Diff, ELO_WinProb_A        ║
║                                                                      ║
║  NaN HANDLING                                                        ║
║    Seeds / WinRate / GamesDiff are NaN for regular-season rows.     ║
║    LightGBM handles NaN natively with dedicated split directions.   ║
║                                                                      ║
║  LEAKAGE CONTROLS                                                    ║
║    • ELO for reg-season rows uses ratings BEFORE each game plays    ║
║    • ELO for tournament rows uses only regular-season game results  ║
║    • 2023 tournament outcomes NEVER update any ELO used as features ║
║    • All other leakage controls from base model preserved           ║
╚══════════════════════════════════════════════════════════════════════╝
"""

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import log_loss, roc_auc_score

print(f"LightGBM version: {lgb.__version__}")

# ─────────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────────────────────────
m_regular_season_detailed = pd.read_csv('/Users/shaurya/Downloads/MRegularSeasonDetailedResults.csv')
m_seeds                   = pd.read_csv('/Users/shaurya/Downloads/MNCAATourneySeeds.csv')
m_tourney_detailed_result = pd.read_csv('/Users/shaurya/Downloads/MNCAATourneyDetailedResults.csv')

print("Regular Season shape :", m_regular_season_detailed.shape)
print("Seeds shape          :", m_seeds.shape)
print("Tourney Results shape:", m_tourney_detailed_result.shape)

# ─────────────────────────────────────────────────────────────────
# 2. PARSE SEEDS
# ─────────────────────────────────────────────────────────────────
def parse_seed(s):
    return int(''.join(filter(str.isdigit, s)))

m_seeds['SeedNum'] = m_seeds['Seed'].apply(parse_seed)

# ─────────────────────────────────────────────────────────────────
# 3. ELO ENGINE
# ─────────────────────────────────────────────────────────────────
BASE_ELO = 1500

def elo_expected(r_a, r_b):
    """Win probability for team A given ratings r_a, r_b."""
    return 1.0 / (1.0 + 10 ** ((r_b - r_a) / 400.0))

def elo_k(mov):
    """Margin-of-victory adjusted K-factor."""
    return 30 if abs(mov) > 10 else 20

def build_elo(reg_df, seasons):
    """
    Walk every regular-season game chronologically within each season.

    Returns:
      pre_game_elo  : dict {(season, daynum, team_id) -> elo BEFORE that game}
      end_of_season : dict {(season, team_id) -> elo at END of regular season}
    """
    pre_game_elo  = {}
    end_of_season = {}

    for season in sorted(seasons):
        games = reg_df[reg_df['Season'] == season].sort_values('DayNum').reset_index(drop=True)
        ratings = {}

        def get_r(tid):
            return ratings.get(tid, BASE_ELO)

        for _, g in games.iterrows():
            w, l = int(g['WTeamID']), int(g['LTeamID'])
            day  = int(g['DayNum'])
            mov  = g['WScore'] - g['LScore']

            r_w, r_l = get_r(w), get_r(l)

            # Store pre-game ratings
            pre_game_elo[(season, day, w)] = r_w
            pre_game_elo[(season, day, l)] = r_l

            # Update ratings
            e_w = elo_expected(r_w, r_l)
            k   = elo_k(mov)
            ratings[w] = r_w + k * (1.0 - e_w)
            ratings[l] = r_l + k * (0.0 - (1.0 - e_w))

        for tid, elo in ratings.items():
            end_of_season[(season, tid)] = elo

    return pre_game_elo, end_of_season

# ─────────────────────────────────────────────────────────────────
# 4. SEASON-AGGREGATED TEAM STATS
# ─────────────────────────────────────────────────────────────────
def compute_team_stats(reg_df, seasons):
    rows = reg_df[reg_df['Season'].isin(seasons)].copy()

    win = rows.rename(columns={
        'WTeamID':'TeamID', 'LTeamID':'OppID',
        'WScore':'Pts',     'LScore':'OppPts',
        'WFGM':'FGM','WFGA':'FGA','WFGM3':'FGM3','WFGA3':'FGA3',
        'WFTM':'FTM','WFTA':'FTA','WOR':'OR','WDR':'DR',
        'WAst':'Ast','WTO':'TO','WStl':'Stl','WBlk':'Blk','WPF':'PF',
    }).copy()
    win['Win'] = 1

    lose = rows.rename(columns={
        'LTeamID':'TeamID', 'WTeamID':'OppID',
        'LScore':'Pts',     'WScore':'OppPts',
        'LFGM':'FGM','LFGA':'FGA','LFGM3':'FGM3','LFGA3':'FGA3',
        'LFTM':'FTM','LFTA':'FTA','LOR':'OR','LDR':'DR',
        'LAst':'Ast','LTO':'TO','LStl':'Stl','LBlk':'Blk','LPF':'PF',
    }).copy()
    lose['Win'] = 0

    keep = ['Season','TeamID','Pts','OppPts','FGM','FGA','FGM3','FGA3',
            'FTM','FTA','OR','DR','Ast','TO','Stl','Blk','PF','Win']
    all_g = pd.concat([win[keep], lose[keep]], ignore_index=True)

    agg = all_g.groupby(['Season','TeamID']).agg(
        Games      =('Win','count'),
        WinRate    =('Win','mean'),
        AvgPts     =('Pts','mean'),
        AvgOppPts  =('OppPts','mean'),
        AvgFGM     =('FGM','mean'),
        AvgFGA     =('FGA','mean'),
        AvgFGM3    =('FGM3','mean'),
        AvgFGA3    =('FGA3','mean'),
        AvgFTM     =('FTM','mean'),
        AvgFTA     =('FTA','mean'),
        AvgOR      =('OR','mean'),
        AvgDR      =('DR','mean'),
        AvgAst     =('Ast','mean'),
        AvgTO      =('TO','mean'),
        AvgStl     =('Stl','mean'),
        AvgBlk     =('Blk','mean'),
        AvgPF      =('PF','mean'),
    ).reset_index()

    agg['FGPct']  = agg['AvgFGM']  / agg['AvgFGA'].replace(0, np.nan)
    agg['FG3Pct'] = agg['AvgFGM3'] / agg['AvgFGA3'].replace(0, np.nan)
    agg['FTPct']  = agg['AvgFTM']  / agg['AvgFTA'].replace(0, np.nan)
    agg['Margin'] = agg['AvgPts']  - agg['AvgOppPts']
    return agg.set_index(['Season','TeamID'])

# ─────────────────────────────────────────────────────────────────
# 5A. REGULAR-SEASON TRAINING ROWS  (with pre-game ELO)
# ─────────────────────────────────────────────────────────────────
def build_reg_season_train_rows(reg_df, seasons, pre_game_elo):
    rows = reg_df[reg_df['Season'].isin(seasons)].copy()
    records = []

    def pct(m, a):
        return m / a if a and a > 0 else np.nan

    for _, g in rows.iterrows():
        w_id, l_id = int(g['WTeamID']), int(g['LTeamID'])
        season, day = int(g['Season']), int(g['DayNum'])

        if w_id < l_id:
            t_a, t_b, label  = w_id, l_id, 1
            pts_a, pts_b     = g['WScore'], g['LScore']
            fgm_a,fga_a      = g['WFGM'],  g['WFGA']
            fgm3_a,fga3_a    = g['WFGM3'], g['WFGA3']
            ftm_a,fta_a      = g['WFTM'],  g['WFTA']
            or_a,dr_a        = g['WOR'],   g['WDR']
            ast_a,to_a       = g['WAst'],  g['WTO']
            stl_a,blk_a,pf_a = g['WStl'], g['WBlk'], g['WPF']
            fgm_b,fga_b      = g['LFGM'],  g['LFGA']
            fgm3_b,fga3_b    = g['LFGM3'], g['LFGA3']
            ftm_b,fta_b      = g['LFTM'],  g['LFTA']
            or_b,dr_b        = g['LOR'],   g['LDR']
            ast_b,to_b       = g['LAst'],  g['LTO']
            stl_b,blk_b,pf_b = g['LStl'], g['LBlk'], g['LPF']
        else:
            t_a, t_b, label  = l_id, w_id, 0
            pts_a, pts_b     = g['LScore'], g['WScore']
            fgm_a,fga_a      = g['LFGM'],  g['LFGA']
            fgm3_a,fga3_a    = g['LFGM3'], g['LFGA3']
            ftm_a,fta_a      = g['LFTM'],  g['LFTA']
            or_a,dr_a        = g['LOR'],   g['LDR']
            ast_a,to_a       = g['LAst'],  g['LTO']
            stl_a,blk_a,pf_a = g['LStl'], g['LBlk'], g['LPF']
            fgm_b,fga_b      = g['WFGM'],  g['WFGA']
            fgm3_b,fga3_b    = g['WFGM3'], g['WFGA3']
            ftm_b,fta_b      = g['WFTM'],  g['WFTA']
            or_b,dr_b        = g['WOR'],   g['WDR']
            ast_b,to_b       = g['WAst'],  g['WTO']
            stl_b,blk_b,pf_b = g['WStl'], g['WBlk'], g['WPF']

        fg_a  = pct(fgm_a,fga_a);   fg_b  = pct(fgm_b,fga_b)
        fg3_a = pct(fgm3_a,fga3_a); fg3_b = pct(fgm3_b,fga3_b)
        ft_a  = pct(ftm_a,fta_a);   ft_b  = pct(ftm_b,fta_b)
        margin_a = pts_a - pts_b

        # Pre-game ELO (before this game is played — no look-ahead)
        elo_a = pre_game_elo.get((season, day, t_a), BASE_ELO)
        elo_b = pre_game_elo.get((season, day, t_b), BASE_ELO)
        elo_prob_a = elo_expected(elo_a, elo_b)

        records.append({
            'Season': season, 'TeamA': t_a, 'TeamB': t_b,
            'Label': label, 'DataSource': 'RegularSeason',

            # ELO features (fully populated for reg-season rows)
            'ELO_A':         elo_a,
            'ELO_B':         elo_b,
            'ELO_Diff':      elo_a - elo_b,
            'ELO_WinProb_A': elo_prob_a,

            # Seeds: NaN for reg-season — LightGBM handles natively
            'SeedA': np.nan, 'SeedB': np.nan, 'SeedDiff': np.nan,

            # Difference features
            'WinRateDiff': np.nan,
            'PtsDiff':     pts_a - pts_b,
            'OppPtsDiff':  pts_b - pts_a,
            'MarginDiff':  margin_a,
            'FGPctDiff':   (fg_a  or 0) - (fg_b  or 0),
            'FG3PctDiff':  (fg3_a or 0) - (fg3_b or 0),
            'FTPctDiff':   (ft_a  or 0) - (ft_b  or 0),
            'ORDiff':      or_a  - or_b,
            'DRDiff':      dr_a  - dr_b,
            'AstDiff':     ast_a - ast_b,
            'TODiff':      to_a  - to_b,
            'StlDiff':     stl_a - stl_b,
            'BlkDiff':     blk_a - blk_b,
            'PFDiff':      pf_a  - pf_b,
            'GamesDiff':   np.nan,

            # Raw Team A
            'A_WinRate': np.nan,  'A_AvgPts': pts_a,   'A_Margin': margin_a,
            'A_FGPct':   fg_a,    'A_FG3Pct': fg3_a,   'A_FTPct':  ft_a,
            'A_AvgOR':   or_a,    'A_AvgDR':  dr_a,
            'A_AvgAst':  ast_a,   'A_AvgTO':  to_a,
            'A_AvgStl':  stl_a,   'A_AvgBlk': blk_a,

            # Raw Team B
            'B_WinRate': np.nan,  'B_AvgPts': pts_b,   'B_Margin': -margin_a,
            'B_FGPct':   fg_b,    'B_FG3Pct': fg3_b,   'B_FTPct':  ft_b,
            'B_AvgOR':   or_b,    'B_AvgDR':  dr_b,
            'B_AvgAst':  ast_b,   'B_AvgTO':  to_b,
            'B_AvgStl':  stl_b,   'B_AvgBlk': blk_b,
        })

    return pd.DataFrame(records)

# ─────────────────────────────────────────────────────────────────
# 5B. TOURNAMENT MATCHUP ROWS  (with end-of-season ELO snapshot)
# ─────────────────────────────────────────────────────────────────
def build_tourney_matchup_rows(tourney_df, stats_df, seeds_df, elo_eos, seasons):
    records = []
    for _, row in tourney_df[tourney_df['Season'].isin(seasons)].iterrows():
        season = row['Season']
        w_id, l_id = row['WTeamID'], row['LTeamID']

        if w_id < l_id:
            t_a, t_b, label = w_id, l_id, 1
        else:
            t_a, t_b, label = l_id, w_id, 0

        if (season, t_a) not in stats_df.index or (season, t_b) not in stats_df.index:
            continue

        sa = stats_df.loc[(season, t_a)]
        sb = stats_df.loc[(season, t_b)]

        s_a = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==t_a)]
        s_b = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==t_b)]
        seed_a = s_a['SeedNum'].values[0] if len(s_a) else np.nan
        seed_b = s_b['SeedNum'].values[0] if len(s_b) else np.nan

        # End-of-regular-season ELO (before tournament starts)
        elo_a = elo_eos.get((season, t_a), BASE_ELO)
        elo_b = elo_eos.get((season, t_b), BASE_ELO)
        elo_prob_a = elo_expected(elo_a, elo_b)

        records.append({
            'Season': season, 'TeamA': t_a, 'TeamB': t_b,
            'Label': label, 'DataSource': 'Tournament',

            # ELO features
            'ELO_A':         elo_a,
            'ELO_B':         elo_b,
            'ELO_Diff':      elo_a - elo_b,
            'ELO_WinProb_A': elo_prob_a,

            # Seeds
            'SeedA': seed_a, 'SeedB': seed_b, 'SeedDiff': seed_a - seed_b,

            # Difference features
            'WinRateDiff':  sa['WinRate']  - sb['WinRate'],
            'PtsDiff':      sa['AvgPts']   - sb['AvgPts'],
            'OppPtsDiff':   sa['AvgOppPts']- sb['AvgOppPts'],
            'MarginDiff':   sa['Margin']   - sb['Margin'],
            'FGPctDiff':    sa['FGPct']    - sb['FGPct'],
            'FG3PctDiff':   sa['FG3Pct']   - sb['FG3Pct'],
            'FTPctDiff':    sa['FTPct']    - sb['FTPct'],
            'ORDiff':       sa['AvgOR']    - sb['AvgOR'],
            'DRDiff':       sa['AvgDR']    - sb['AvgDR'],
            'AstDiff':      sa['AvgAst']   - sb['AvgAst'],
            'TODiff':       sa['AvgTO']    - sb['AvgTO'],
            'StlDiff':      sa['AvgStl']   - sb['AvgStl'],
            'BlkDiff':      sa['AvgBlk']   - sb['AvgBlk'],
            'PFDiff':       sa['AvgPF']    - sb['AvgPF'],
            'GamesDiff':    sa['Games']    - sb['Games'],

            # Raw Team A
            'A_WinRate': sa['WinRate'], 'A_AvgPts': sa['AvgPts'],
            'A_Margin':  sa['Margin'],  'A_FGPct':  sa['FGPct'],
            'A_FG3Pct':  sa['FG3Pct'], 'A_FTPct':  sa['FTPct'],
            'A_AvgOR':   sa['AvgOR'],   'A_AvgDR':  sa['AvgDR'],
            'A_AvgAst':  sa['AvgAst'], 'A_AvgTO':   sa['AvgTO'],
            'A_AvgStl':  sa['AvgStl'], 'A_AvgBlk':  sa['AvgBlk'],

            # Raw Team B
            'B_WinRate': sb['WinRate'], 'B_AvgPts': sb['AvgPts'],
            'B_Margin':  sb['Margin'],  'B_FGPct':  sb['FGPct'],
            'B_FG3Pct':  sb['FG3Pct'], 'B_FTPct':  sb['FTPct'],
            'B_AvgOR':   sb['AvgOR'],   'B_AvgDR':  sb['AvgDR'],
            'B_AvgAst':  sb['AvgAst'], 'B_AvgTO':   sb['AvgTO'],
            'B_AvgStl':  sb['AvgStl'], 'B_AvgBlk':  sb['AvgBlk'],
        })
    return pd.DataFrame(records)

# ─────────────────────────────────────────────────────────────────
# 6. ASSEMBLE TRAIN / TEST
# ─────────────────────────────────────────────────────────────────
REG_TRAIN_SEASONS   = list(range(2013, 2024))   # 2013–2023 regular seasons
TOURN_TRAIN_SEASONS = list(range(2013, 2023))   # 2013–2022 tournaments
TEST_SEASONS        = [2023]
ALL_SEASONS         = list(range(2013, 2024))

print("\nBuilding ELO ratings (chronological, per season)...")
pre_game_elo, elo_end_of_season = build_elo(m_regular_season_detailed, ALL_SEASONS)
print(f"  Pre-game ELO entries    : {len(pre_game_elo):,}")
print(f"  End-of-season snapshots : {len(elo_end_of_season):,}")

elo_2023 = {k: v for k, v in elo_end_of_season.items() if k[0] == 2023}
top5 = sorted(elo_2023.items(), key=lambda x: -x[1])[:5]
print("\nTop 5 ELO ratings at end of 2023 regular season:")
for (s, t), e in top5:
    print(f"  Season {s} | TeamID {t:4d} | ELO {e:.1f}")

stats = compute_team_stats(m_regular_season_detailed, ALL_SEASONS)

reg_train_df   = build_reg_season_train_rows(
    m_regular_season_detailed, REG_TRAIN_SEASONS, pre_game_elo)
tourn_train_df = build_tourney_matchup_rows(
    m_tourney_detailed_result, stats, m_seeds, elo_end_of_season, TOURN_TRAIN_SEASONS)
train_df       = pd.concat([reg_train_df, tourn_train_df], ignore_index=True)
test_df        = build_tourney_matchup_rows(
    m_tourney_detailed_result, stats, m_seeds, elo_end_of_season, TEST_SEASONS)

print(f"\n{'─'*55}")
print(f"  Training rows")
print(f"{'─'*55}")
print(f"  Regular season games  2013–2023 : {len(reg_train_df):>7,}")
print(f"  Tournament games      2013–2022 : {len(tourn_train_df):>7,}")
print(f"  TOTAL                           : {len(train_df):>7,}")
print(f"\n  Test rows (2023 tournament)     : {len(test_df):>7,}")
print(f"{'─'*55}")

# ─────────────────────────────────────────────────────────────────
# 7. FEATURE COLUMNS  (42 base + 4 ELO = 46 total)
# ─────────────────────────────────────────────────────────────────
FEATURE_COLS = [
    # ELO features (4) — fully populated for ALL rows
    'ELO_A', 'ELO_B', 'ELO_Diff', 'ELO_WinProb_A',

    # Seeds (NaN for reg-season rows — LightGBM handles natively)
    'SeedA', 'SeedB', 'SeedDiff',

    # Difference features
    'WinRateDiff', 'PtsDiff', 'OppPtsDiff', 'MarginDiff',
    'FGPctDiff', 'FG3PctDiff', 'FTPctDiff',
    'ORDiff', 'DRDiff', 'AstDiff', 'TODiff', 'StlDiff', 'BlkDiff', 'PFDiff',
    'GamesDiff',

    # Raw Team A
    'A_WinRate', 'A_AvgPts', 'A_Margin', 'A_FGPct', 'A_FG3Pct', 'A_FTPct',
    'A_AvgOR', 'A_AvgDR', 'A_AvgAst', 'A_AvgTO', 'A_AvgStl', 'A_AvgBlk',

    # Raw Team B
    'B_WinRate', 'B_AvgPts', 'B_Margin', 'B_FGPct', 'B_FG3Pct', 'B_FTPct',
    'B_AvgOR', 'B_AvgDR', 'B_AvgAst', 'B_AvgTO', 'B_AvgStl', 'B_AvgBlk',
]

X_train = train_df[FEATURE_COLS]
y_train = train_df['Label']
X_test  = test_df[FEATURE_COLS]
y_test  = test_df['Label']

print(f"\n  Total features : {len(FEATURE_COLS)} (42 base + 4 ELO)")
print(f"  ELO features   : ELO_A, ELO_B, ELO_Diff, ELO_WinProb_A")
print(f"  NaN cols (Seed*, WinRate*, GamesDiff) handled natively by LightGBM")

# ─────────────────────────────────────────────────────────────────
# 8. TRAIN LIGHTGBM
#    Hyperparameters mirror the base LightGBM Script 1 model for
#    consistency across the model family.
# ─────────────────────────────────────────────────────────────────
lgb_train_ds = lgb.Dataset(X_train, label=y_train)
lgb_valid_ds = lgb.Dataset(X_test,  label=y_test, reference=lgb_train_ds)

params = {
    'objective':         'binary',
    'metric':            'binary_logloss',
    'boosting_type':     'gbdt',
    'num_leaves':        31,
    'learning_rate':     0.05,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'min_child_samples': 20,
    'lambda_l1':         0.1,
    'lambda_l2':         0.1,
    'use_missing':       True,  # explicit NaN splits for seed/winrate cols
    'verbose':           -1,
    'seed':              42,
}

model = lgb.train(
    params,
    lgb_train_ds,
    num_boost_round=600,
    valid_sets=[lgb_valid_ds],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(50)],
)

best_score = model.best_score['valid_0']['binary_logloss']
print(f"\nBest iteration  : {model.best_iteration}")
print(f"Best val logloss: {best_score:.6f}")

# ─────────────────────────────────────────────────────────────────
# 9. EVALUATE ON 2023 TOURNAMENT
# ─────────────────────────────────────────────────────────────────
y_pred_prob = model.predict(X_test)
y_pred_bin  = (y_pred_prob >= 0.5).astype(int)

ll  = log_loss(y_test, y_pred_prob)
auc = roc_auc_score(y_test, y_pred_prob)
acc = (y_pred_bin == y_test.values).mean()

print(f"\n{'='*55}")
print(f"  LightGBM + ELO — 2023 Tournament Evaluation")
print(f"  Train: reg season 2013-2023 + tournaments 2013-2022")
print(f"  Test : 2023 tournament only")
print(f"{'='*55}")
print(f"  Log Loss : {ll:.4f}   (random baseline = 0.6931)")
print(f"  ROC AUC  : {auc:.4f}   (random baseline = 0.5000)")
print(f"  Accuracy : {acc:.4f}   ({int(acc*len(y_test))}/{len(y_test)} correct)")

# ─────────────────────────────────────────────────────────────────
# 10. ELO-ONLY BASELINE
# ─────────────────────────────────────────────────────────────────
elo_probs = test_df['ELO_WinProb_A'].values
elo_ll    = log_loss(y_test, elo_probs)
elo_auc   = roc_auc_score(y_test, elo_probs)
elo_acc   = ((elo_probs >= 0.5).astype(int) == y_test.values).mean()

print(f"\n{'─'*55}")
print(f"  ELO-ONLY BASELINE (for reference)")
print(f"{'─'*55}")
print(f"  Log Loss : {elo_ll:.4f}")
print(f"  ROC AUC  : {elo_auc:.4f}")
print(f"  Accuracy : {elo_acc:.4f}   ({int(elo_acc*len(y_test))}/{len(y_test)} correct)")
print(f"\n  LightGBM+ELO improvement over ELO-only:")
print(f"  ΔLog Loss : {elo_ll - ll:+.4f}  (positive = LightGBM+ELO better)")
print(f"  ΔAUC      : {auc - elo_auc:+.4f}  (positive = LightGBM+ELO better)")

# ─────────────────────────────────────────────────────────────────
# 11. FEATURE IMPORTANCE  (gain — consistent with XGBoost model)
# ─────────────────────────────────────────────────────────────────
importance = pd.DataFrame({
    'Feature':    model.feature_name(),
    'Importance': model.feature_importance(importance_type='gain'),
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("\nTop 15 Features by Gain:")
print(importance.head(15).to_string(index=False))

# ELO feature ranks
elo_feats = ['ELO_A','ELO_B','ELO_Diff','ELO_WinProb_A']
print("\nELO feature ranks:")
for feat in elo_feats:
    row = importance[importance['Feature'] == feat]
    if not row.empty:
        rank = row.index[0] + 1
        gain = row['Importance'].values[0]
        print(f"  Rank {rank:>2d} | {feat:<16s} | Gain {gain:.3f}")

# ─────────────────────────────────────────────────────────────────
# 12. OUTPUT
# ─────────────────────────────────────────────────────────────────
results_df = test_df[['Season','TeamA','TeamB','Label'] + FEATURE_COLS].copy()
results_df['PredProb_TeamA_Wins'] = y_pred_prob
results_df['PredWinner']   = results_df.apply(
    lambda r: r['TeamA'] if r['PredProb_TeamA_Wins'] >= 0.5 else r['TeamB'], axis=1)
results_df['ActualWinner'] = results_df.apply(
    lambda r: r['TeamA'] if r['Label'] == 1 else r['TeamB'], axis=1)
results_df['Correct']      = (results_df['PredWinner'] == results_df['ActualWinner']).astype(int)
results_df['FeaturesUsed'] = ' | '.join(FEATURE_COLS)

results_df.to_csv('/Users/shaurya/Downloads/lgbm_elo_2023_predictions.csv', index=False)
importance.to_csv('/Users/shaurya/Downloads/lgbm_elo_2023_feature_importance.csv', index=False)

print(f"\n✅  Predictions saved  → /Users/shaurya/Downloads/lgbm_elo_2023_predictions.csv")
print(f"✅  Importance saved   → /Users/shaurya/Downloads/lgbm_elo_2023_feature_importance.csv")
print(f"\nTotal features : {len(FEATURE_COLS)}  (42 base + 4 ELO)")
print("Feature list   :", FEATURE_COLS)

LightGBM version: 4.6.0
Regular Season shape : (122775, 34)
Seeds shape          : (2626, 3)
Tourney Results shape: (1449, 34)

Building ELO ratings (chronological, per season)...
  Pre-game ELO entries    : 115,596
  End-of-season snapshots : 3,876

Top 5 ELO ratings at end of 2023 regular season:
  Season 2023 | TeamID 1222 | ELO 1747.4
  Season 2023 | TeamID 1104 | ELO 1736.0
  Season 2023 | TeamID 1417 | ELO 1725.7
  Season 2023 | TeamID 1194 | ELO 1723.1
  Season 2023 | TeamID 1266 | ELO 1714.2

───────────────────────────────────────────────────────
  Training rows
───────────────────────────────────────────────────────
  Regular season games  2013–2023 :  57,798
  Tournament games      2013–2022 :     602
  TOTAL                           :  58,400

  Test rows (2023 tournament)     :      67
───────────────────────────────────────────────────────

  Total features : 46 (42 base + 4 ELO)
  ELO features   : ELO_A, ELO_B, ELO_Diff, ELO_WinProb_A
  NaN cols (Seed*, WinRate*, GamesD